In [2]:
import pickle

lattices = pickle.load(open("/home/zhang/Metamaterial-Benchmark/unit_cell_catalog/train_set.pkl", "rb"))

lattices[0]['edge_index'].shape

(2, 106)

In [1]:
import pickle
import os
import gzip
import numpy as np

def create_cif_content(lattice, index):
    """Convert a lattice dictionary into a CIF file format."""
    
    # Extract the lattice parameters
    lengths = lattice['lengths'][0]  
    angles = lattice['angles'][0]    
    frac_coords = lattice['frac_coords']
    edge_index = lattice['edge_index'].T
    
    y = lattice['y'][0]
    youngs_modulus = y[0:3]
    shear_modulus = y[3:6]
    poisson_ratio = y[6:12]
    
    structure_id = f"structure_{index}"
    
    cif_content = f"""lattice_
data_

_cell_length_a   {lengths[0]:.6f}
_cell_length_b   {lengths[1]:.6f}
_cell_length_c   {lengths[2]:.6f}
_cell_angle_alpha   {angles[0]:.6f}
_cell_angle_beta    {angles[1]:.6f}
_cell_angle_gamma   {angles[2]:.6f}

loop_
_atom_index
_atom_site_fract_x
_atom_site_fract_y
_atom_site_fract_z
"""

    for i, coord in enumerate(frac_coords):
        cif_content += f"atom_{i}  {coord[0]:.6f}  {coord[1]:.6f}  {coord[2]:.6f}\n"
    
    cif_content += "\n\nloop_\n_edge_from_atom\n_edge_to_atom\n"
    for i in range(len(edge_index)):
        from_atom = edge_index[i][0]
        to_atom = edge_index[i][1]
        cif_content += f"{from_atom}  {to_atom}\n"
    
    return structure_id, cif_content

# Number of augmentation rounds
n_augment_rounds = 10

# Load the lattices
lattices = pickle.load(open("/home/zhang/Metamaterial-Benchmark/unit_cell_catalog/train_set.pkl", "rb"))

cif_data = []
structure_count = 0

for round_idx in range(n_augment_rounds):
    for i, lattice in enumerate(lattices):
        # Copy the lattice to avoid in-place modifications
        lattice_copy = dict(lattice)
        
        frac_coords = lattice_copy['frac_coords']
        edge_index = lattice_copy['edge_index'].T
        
        # Shuffle the atom order
        num_atoms = len(frac_coords)
        permutation = np.random.permutation(num_atoms)
        new_frac_coords = frac_coords[permutation]
        
        # Map old indices to new indices
        old_to_new = {old_idx: new_idx for new_idx, old_idx in enumerate(permutation)}
        
        # Update edges
        new_edge_index = []
        for edge in edge_index:
            old_from, old_to = edge
            new_from = old_to_new[old_from]
            new_to = old_to_new[old_to]
            new_edge_index.append([new_from, new_to])
        new_edge_index = np.array(new_edge_index, dtype=int)
        
        # Optionally shuffle the edges as well
        edge_perm = np.random.permutation(len(new_edge_index))
        new_edge_index = new_edge_index[edge_perm]
        
        # Store the new data back
        lattice_copy['frac_coords'] = new_frac_coords
        lattice_copy['edge_index'] = new_edge_index.T
        
        structure_id, cif_content = create_cif_content(lattice_copy, f"{round_idx}_{structure_count}")
        cif_data.append((structure_id, cif_content))
        structure_count += 1

# Save as compressed pickle file
with gzip.open('/home/zhang/Metamaterial-Benchmark/crystaLLM/aug_train_structures.pkl.gz', 'wb') as f:
    pickle.dump(cif_data, f)


In [2]:
import gzip
import pickle

with gzip.open('/home/zhang/Metamaterial-Benchmark/crystaLLM/aug_train_structures.pkl.gz', 'rb') as f:
    lattice_cif = pickle.load(f)
lattice_cif[0]

('structure_0_0',
 'lattice_\ndata_\n\n_cell_length_a   0.932480\n_cell_length_b   0.813210\n_cell_length_c   1.000000\n_cell_angle_alpha   90.000000\n_cell_angle_beta    115.300003\n_cell_angle_gamma   90.000000\n\nloop_\n_atom_index\n_atom_site_fract_x\n_atom_site_fract_y\n_atom_site_fract_z\natom_0  0.894820  0.745160  0.000000\natom_1  0.669500  0.829480  0.868040\natom_2  0.000000  0.599620  0.732560\natom_3  1.000000  0.294210  0.938400\natom_4  0.144670  1.000000  0.899690\natom_5  0.855850  1.000000  0.146940\natom_6  0.000000  0.205790  0.561600\natom_7  0.630030  0.123660  0.000000\natom_8  0.855330  0.000000  0.100310\natom_9  1.000000  0.750000  0.750000\natom_10  0.000000  0.746740  0.082650\natom_11  1.000000  0.681440  0.064310\natom_12  0.000000  0.250000  0.250000\natom_13  0.855330  1.000000  0.100310\natom_14  0.000000  0.318560  0.935690\natom_15  0.630030  0.123660  1.000000\natom_16  0.837300  0.531560  0.602510\natom_17  1.000000  0.818560  0.435690\natom_18  0.3

In [19]:
import numpy as np
import pickle
from typing import List, Dict, Tuple
import os
from pathlib import Path

def parse_cif_file(file_path: str) -> Dict:
    """Parse a single .cif file and return its parameters."""
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    structure = {
        'lengths': [],
        'angles': [],
        'coordinates': [],
        'edge_index': []
    }
    
    # Parse lattice parameters
    for line in lines:
        line = line.strip()
        if not line:
            continue
            
        parts = line.split()
        if len(parts) < 2:
            continue
            
        try:
            if line.startswith('_cell_length'):
                value = float(parts[-1])
                structure['lengths'].append(value)
            elif line.startswith('_cell_angle'):
                value = float(parts[-1])
                structure['angles'].append(value)
        except ValueError:
            continue
    
    # Parse atomic coordinates
    coord_section = False
    for line in lines:
        if line.startswith('loop_'):
            coord_section = True
            continue
        if coord_section and line.startswith('atom_'):
            parts = line.split()
            try:
                coords = [float(x) for x in parts[1:4]]
                structure['coordinates'].append(coords)
            except (ValueError, IndexError):
                continue
        elif coord_section and line.startswith('_edge'):
            coord_section = False
    
    # Parse edge indices
    edge_section = False
    for line in lines:
        if '_edge_from_atom' in line:
            edge_section = True
            continue
        if edge_section and line.strip() and not line.startswith('loop_'):
            try:
                from_atom, to_atom = map(int, line.split())
                structure['edge_index'].append([from_atom, to_atom])
            except ValueError:
                continue
    
    # Convert to numpy arrays
    structure['lengths'] = np.array(structure['lengths'])
    structure['angles'] = np.array(structure['angles'])
    structure['coordinates'] = np.array(structure['coordinates'])
    structure['edge_index'] = np.array(structure['edge_index'])
    print(structure['edge_index'].T.shape[0])
    # Verify we have the correct number of parameters
    if len(structure['lengths']) != 3 or len(structure['angles']) != 3:
        raise ValueError("Missing lattice parameters")
    
    return structure

def process_directory(directory_path: str, output_file: str):
    """Process all .cif files in the directory and save to a pickle file."""
    structures = []
    
    # Get all .cif files in the directory
    cif_files = list(Path(directory_path).glob('*.cif'))
    
    print(f"Found {len(cif_files)} .cif files")
    
    # Process each file
    for file_path in cif_files:
        try:
            structure = parse_cif_file(str(file_path))
            structures.append(structure)  # Use filename without extension as key
            print(f"Processed {file_path.name}")
        except Exception as e:
            print(f"Error processing {file_path.name}: {str(e)}")
    
    # Save to pickle file
    with open(output_file, 'wb') as f:
        pickle.dump(structures, f)
    
    print(f"\nSaved {len(structures)} structures to {output_file}")

if __name__ == "__main__":
    # Directory containing .cif files
    directory = "/home/zhang/CrystaLLM/gen_lattices"  # Change this to your directory path
    output_file = "structures.pkl"
    
    process_directory(directory, output_file)

Found 1000 .cif files
2
Processed sample_16.cif
2
Processed sample_741.cif
2
Error processing sample_320.cif: Missing lattice parameters
2
Processed sample_487.cif
2
Processed sample_456.cif
2
Processed sample_824.cif
2
Processed sample_790.cif
2
Processed sample_619.cif
2
Processed sample_278.cif
2
Processed sample_889.cif
2
Processed sample_858.cif
2
Processed sample_900.cif
2
Processed sample_572.cif
2
Processed sample_113.cif
2
Processed sample_665.cif
2
Error processing sample_204.cif: Missing lattice parameters
2
Processed sample_575.cif
2
Processed sample_907.cif
2
Error processing sample_114.cif: Missing lattice parameters
2
Processed sample_662.cif
2
Processed sample_203.cif
2
Processed sample_11.cif
2
Processed sample_746.cif
0
Processed sample_480.cif
2
Error processing sample_327.cif: Missing lattice parameters
2
Processed sample_823.cif
2
Processed sample_451.cif
2
Processed sample_797.cif
2
Processed sample_509.cif
2
Processed sample_168.cif
2
Processed sample_63.cif
2
Pr

In [22]:
import pickle

# Load the pickle file
with open('/home/zhang/Metamaterial-Benchmark/crystaLLM/structures.pkl', 'rb') as f:
    structures = pickle.load(f)

first_structure = structures[0]
print(len(structures))


948


In [21]:
import pickle

# Load the test set pickle file
with open('/home/zhang/Metamaterial-Benchmark/gen_samples_crystal-text-llm/generated_structures.pkl', 'rb') as f:
    test_data = pickle.load(f)

# Get and print the first element
first_test_item = test_data[0]
print(first_test_item)


{'lengths': array([1.    , 1.    , 0.5655]), 'angles': array([90., 90., 90.]), 'coordinates': array([[0.115, 0.385, 0.25 ],
       [0.115, 0.615, 0.25 ],
       [0.385, 0.115, 0.25 ],
       [0.615, 0.115, 0.25 ],
       [0.385, 0.885, 0.25 ],
       [0.615, 0.885, 0.25 ],
       [0.885, 0.385, 0.25 ],
       [0.885, 0.615, 0.25 ],
       [0.115, 0.385, 0.75 ],
       [0.115, 0.615, 0.75 ],
       [0.385, 0.115, 0.75 ],
       [0.615, 0.115, 0.75 ],
       [0.385, 0.885, 0.75 ],
       [0.615, 0.885, 0.75 ],
       [0.885, 0.385, 0.75 ],
       [0.885, 0.615, 0.75 ],
       [0.115, 0.385, 0.   ],
       [0.385, 0.115, 0.   ],
       [0.115, 0.615, 0.   ],
       [0.615, 0.115, 0.   ],
       [0.385, 0.885, 0.   ],
       [0.885, 0.385, 0.   ],
       [0.615, 0.885, 0.   ],
       [0.885, 0.615, 0.   ],
       [0.115, 0.385, 1.   ],
       [0.385, 0.115, 1.   ],
       [0.115, 0.615, 1.   ],
       [0.615, 0.115, 1.   ],
       [0.385, 0.885, 1.   ],
       [0.885, 0.385, 1.   ],
      